In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('uniprotkb_Homo_sapiens_human_AND_model_2026_03_08.tsv', sep='\t')


In [ ]:
df.head()

,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Function [CC],Pathway,UniPathway,PathwayCommons,Subcellular location [CC]
0,A0A087X1C5,reviewed,CP2D7_HUMAN,Cytochrome P450 2D7 (EC 1.14.14.1),CYP2D7,Homo sapiens (Human),515,FUNCTION: May be responsible for the metabolis...,NaN,NaN,A0A087X1C5;,SUBCELLULAR LOCATION: Membrane {ECO:0000305}; ...
1,A0A096LP01,reviewed,SIM26_HUMAN,Small integral membrane protein 26,SMIM26 LINC00493,Homo sapiens (Human),95,FUNCTION: May play a role in cell viability. {...,NaN,NaN,NaN,SUBCELLULAR LOCATION: Mitochondrion outer memb...
2,A0A0B4J2F0,reviewed,PIOS1_HUMAN,Protein PIGBOS1 (PIGB opposite strand protein 1),PIGBOS1,Homo sapiens (Human),54,FUNCTION: Plays a role in regulation of the un...,NaN,NaN,A0A0B4J2F0;,SUBCELLULAR LOCATION: Mitochondrion outer memb...
3,A0A0C5B5G6,reviewed,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),16,FUNCTION: Regulates insulin sensitivity and me...,NaN,NaN,NaN,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...
4,A0A0K2S4Q6,reviewed,CD3CH_HUMAN,Protein CD300H (CD300 antigen-like family memb...,CD300H,Homo sapiens (Human),201,FUNCTION: May play an important role in innate...,NaN,NaN,A0A0K2S4Q6;,SUBCELLULAR LOCATION: [Isoform 1]: Membrane {E...


In [ ]:
df.shape

(20431, 12)

In [ ]:
df["Function [CC]"].iloc[9]

'FUNCTION: Cation/chloride cotransporter that may play a role in the control of keratinocyte proliferation. {ECO:0000269|PubMed:11863360}.'

In [ ]:
df["clean_function"] = (
    df["Function [CC]"]
    .str.replace(r"^FUNCTION:\s*", "", regex=True)
    .str.replace(r"\{[^}]*\}", "", regex=True)
    .str.replace(r"\([^)]*\)", "", regex=True)
    .str.replace(r"\[[^\]]*\]", "", regex=True)
    .str.strip()
)

In [ ]:
df["clean_function"].iloc[5]

'Promotes dispersal of P-body components and is likely to play a role in the mRNA decapping process. .'

In [ ]:
df = df.dropna(subset=["clean_function"])

In [ ]:
df.shape

(17259, 13)

In [ ]:
df = df.drop_duplicates()


In [ ]:
df.shape

(17259, 13)

In [ ]:
df["clean_function"] = df["clean_function"].str.lower()


In [ ]:
df["tokens"] = df["clean_function"].apply(word_tokenize)


In [ ]:
stop_words = set(stopwords.words("english"))

df["tokens"] = df["tokens"].apply(
    lambda x: [word for word in x if word not in stop_words]
)

In [ ]:
lemmatizer = WordNetLemmatizer()

df["tokens"] = df["tokens"].apply(
    lambda x: [lemmatizer.lemmatize(word) for word in x]
)

In [ ]:
df["processed_text"] = df["tokens"].apply(lambda x: " ".join(x))


In [ ]:
df["processed_text"]

,processed_text
0,may responsible metabolism many drug environme...
1,may play role cell viability . .
2,play role regulation unfolded protein response...
3,regulates insulin sensitivity metabolic homeos...
4,may play important role innate immunity mediat...
...,...
20082,play role neuroprotective antiapoptotic factor...
20083,play role neuroprotective antiapoptotic factor...
20136,tooth-associated epithelium protein may partic...
20238,precursor cornified envelope stratum corneum . .


In [ ]:
df.to_csv("processed_gene_function_dataset.csv", index=False)